# Quantum-Simulated LLM — Google Colab (T4 GPU)
**World's first hybrid quantum-classical language model with agentic workflows**

Author: Saikrishna Garikipati | [GitHub](https://github.com/saikrishnajava/quantum-llm-agent)

> **Hit Runtime → Change runtime type → T4 GPU, then Runtime → Run all**

This notebook runs end-to-end unattended. Total estimated time: ~15-25 minutes.

In [ ]:
##############################################################################
# STEP 1: SETUP — Clone, install, verify GPU
##############################################################################
import subprocess, sys, os, time

print('=' * 60)
print('STEP 1: SETUP')
print('=' * 60)

# Clone
if not os.path.exists('quantum-llm-agent'):
    !git clone -q https://github.com/saikrishnajava/quantum-llm-agent.git
    print('Cloned repo.')
else:
    print('Repo already exists.')

os.chdir('quantum-llm-agent')
sys.path.insert(0, '.')

# Install
!pip install -q pennylane pennylane-lightning numpy scipy pyyaml pytest 2>&1 | tail -1
!pip install -q pennylane-lightning[gpu] custatevec-cu12 2>/dev/null || echo 'GPU packages unavailable, will use CPU fallback'
!pip install -q -e . 2>&1 | tail -1

# Verify GPU
print()
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader 2>/dev/null || echo 'No NVIDIA GPU detected'

# Detect best backend
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pennylane as qml

BACKEND = 'default.qubit'
DIFF_METHOD = 'backprop'
for name, dm in [('lightning.gpu', 'adjoint'), ('lightning.qubit', 'adjoint')]:
    try:
        dev = qml.device(name, wires=2)
        @qml.qnode(dev)
        def _t(): qml.Hadamard(0); return qml.expval(qml.PauliZ(0))
        _t()
        BACKEND, DIFF_METHOD = name, dm
        break
    except: pass

print(f'PennyLane {qml.__version__} | Backend: {BACKEND} | Diff: {DIFF_METHOD}')
print('Setup complete.')

In [ ]:
##############################################################################
# STEP 2: RUN TESTS
##############################################################################
print('=' * 60)
print('STEP 2: TEST SUITE')
print('=' * 60)
!python -m pytest tests/ -v --tb=short 2>&1 | tail -15

In [ ]:
##############################################################################
# STEP 3: SHARED SETUP — Corpus, tokenizer, data (used by all steps below)
##############################################################################
from hybrid.interfaces.model import HybridQuantumLLM
from classical.optimizers.trainer import HybridQuantumTrainer
from classical.tokenizer import CharTokenizer
from classical.data import TextDataset, DataLoader

CORPUS = '''To be or not to be that is the question
Whether tis nobler in the mind to suffer
The slings and arrows of outrageous fortune
Or to take arms against a sea of troubles
And by opposing end them To die to sleep
No more and by a sleep to say we end
The heartache and the thousand natural shocks
That flesh is heir to Tis a consummation
Devoutly to be wished To die to sleep
To sleep perchance to dream ay there s the rub
For in that sleep of death what dreams may come
When we have shuffled off this mortal coil
Must give us pause There s the respect
That makes calamity of so long life
All that glitters is not gold
The fault dear Brutus is not in our stars
But in ourselves that we are underlings
Friends Romans countrymen lend me your ears
I come to bury Caesar not to praise him
The evil that men do lives after them
The good is oft interred with their bones
Now is the winter of our discontent
Made glorious summer by this sun of York
A horse a horse my kingdom for a horse
Out out brief candle life is but a walking shadow
A poor player that struts and frets his hour upon the stage
And then is heard no more it is a tale told by an idiot
Full of sound and fury signifying nothing
Double double toil and trouble fire burn and cauldron bubble
Something wicked this way comes
We are such stuff as dreams are made on
And our little life is rounded with a sleep'''.strip()

tokenizer = CharTokenizer.from_text(CORPUS)
token_ids = tokenizer.encode(CORPUS)
print(f'Corpus: {len(CORPUS)} chars | Vocab: {tokenizer.vocab_size} | Tokens: {len(token_ids)}')

# Helper to train any model
def train_model(model, seq_len, batch_size, n_epochs, lr=1e-3):
    ds = TextDataset(token_ids, seq_len)
    dl = DataLoader(ds, batch_size, shuffle=True)
    trainer = HybridQuantumTrainer(model, learning_rate=lr)
    params = model.count_parameters()
    print(f'Model: {params["total"]:,} params ({params["quantum"]} quantum) | {len(dl)} batches/epoch')
    print(f'Training {n_epochs} epochs...')
    t_total = time.perf_counter()
    epoch_data = []
    for epoch in range(n_epochs):
        t0 = time.perf_counter()
        losses = []
        for x, y in dl:
            result = trainer.training_step(x, y)
            losses.append(result['loss'])
        elapsed = time.perf_counter() - t0
        avg_loss = np.mean(losses)
        ms_step = elapsed / len(losses) * 1000
        epoch_data.append({'loss': avg_loss, 'ms_step': ms_step, 'time': elapsed})
        print(f'  Epoch {epoch+1:2d}: loss={avg_loss:.4f} ({ms_step:.0f}ms/step, {elapsed:.1f}s)')
    total = time.perf_counter() - t_total
    print(f'Done in {total:.1f}s ({total/60:.1f} min) | Loss: {epoch_data[0]["loss"]:.4f} -> {epoch_data[-1]["loss"]:.4f}')
    return model, epoch_data

# Helper to generate text
def generate(model, prompts, max_tokens=60, temp=0.7):
    model.eval()
    for p in prompts:
        ids = np.array([tokenizer.encode(p)])
        gen = model.generate(ids, max_new_tokens=max_tokens, temperature=temp, do_sample=True)
        print(f'  "{p}" -> "{tokenizer.decode(gen[0])}"')

print('Shared setup ready.')

In [ ]:
##############################################################################
# STEP 4: QUICK BENCHMARK — 10 steps, compare to Mac/Linux
##############################################################################
print('=' * 60)
print('STEP 4: QUICK BENCHMARK (10 steps)')
print('=' * 60)

bench_model = HybridQuantumLLM(
    vocab_size=tokenizer.vocab_size, d_model=32, n_layers=1, n_heads=4,
    quantum_heads_per_layer=1, d_ff=64, max_seq_length=32, dropout=0.0,
    quantum_config={'use_quantum_embedding': False, 'embedding_qubits': 3,
                    'attention_qubits': 6, 'activation_qubits': 3,
                    'use_quantum_activation': False})

bench_trainer = HybridQuantumTrainer(bench_model, learning_rate=1e-3)
ds = TextDataset(token_ids, 8)
dl = DataLoader(ds, 8, shuffle=True)

times = []
for i, (x, y) in enumerate(dl):
    if i >= 10: break
    t0 = time.perf_counter()
    r = bench_trainer.training_step(x, y)
    ms = (time.perf_counter() - t0) * 1000
    times.append(ms)
    print(f'  Step {i+1}: loss={r["loss"]:.4f}, {ms:.0f}ms')

avg = np.mean(times[1:])
print(f'\n{"=" * 40}')
print(f'Colab T4:     {avg:.0f} ms/step')
print(f'Linux CPU:    2450 ms/step  (Colab is {2450/avg:.1f}x faster)')
print(f'MacBook CPU:  3534 ms/step  (Colab is {3534/avg:.1f}x faster)')
del bench_model, bench_trainer

In [ ]:
##############################################################################
# STEP 5: STANDARD MODEL — 10 epochs on Shakespeare
##############################################################################
print('=' * 60)
print('STEP 5: STANDARD MODEL (d=32, 1 quantum head, 10 epochs)')
print('=' * 60)

std_model = HybridQuantumLLM(
    vocab_size=tokenizer.vocab_size, d_model=32, n_layers=1, n_heads=4,
    quantum_heads_per_layer=1, d_ff=64, max_seq_length=32, dropout=0.0,
    quantum_config={'use_quantum_embedding': False, 'embedding_qubits': 3,
                    'attention_qubits': 6, 'activation_qubits': 3,
                    'use_quantum_activation': False})

std_model, std_data = train_model(std_model, seq_len=8, batch_size=8, n_epochs=10)

print('\nGeneration:')
generate(std_model, ['To be', 'The fault', 'Now is', 'A horse'])

In [ ]:
##############################################################################
# STEP 6: BIGGER MODEL — d=64, 2 layers, 2 quantum heads, quantum embedding
##############################################################################
print('=' * 60)
print('STEP 6: BIGGER MODEL (d=64, 2 layers, 2 qheads, quantum embed)')
print('=' * 60)

big_model = HybridQuantumLLM(
    vocab_size=tokenizer.vocab_size, d_model=64, n_layers=2, n_heads=8,
    quantum_heads_per_layer=2, d_ff=256, max_seq_length=64, dropout=0.0,
    quantum_config={'use_quantum_embedding': True, 'embedding_qubits': 4,
                    'attention_qubits': 6, 'activation_qubits': 3,
                    'use_quantum_activation': False})

big_model, big_data = train_model(big_model, seq_len=16, batch_size=4, n_epochs=10, lr=5e-4)

print('\nGeneration:')
generate(big_model, ['To be or not', 'The fault dear', 'Something wicked'])

In [ ]:
##############################################################################
# STEP 7: T4 STRESS TEST — d=128, 4 layers, max quantum
##############################################################################
print('=' * 60)
print('STEP 7: T4 STRESS TEST (d=128, 4 layers, 6-qubit embed)')
print('=' * 60)

max_model = HybridQuantumLLM(
    vocab_size=tokenizer.vocab_size, d_model=128, n_layers=4, n_heads=8,
    quantum_heads_per_layer=2, d_ff=512, max_seq_length=64, dropout=0.0,
    quantum_config={'use_quantum_embedding': True, 'embedding_qubits': 6,
                    'attention_qubits': 6, 'activation_qubits': 4,
                    'use_quantum_activation': False})

max_params = max_model.count_parameters()
print(f'Model: {max_params["total"]:,} params ({max_params["quantum"]} quantum)')

# Forward pass timing
test_ids = np.random.randint(0, tokenizer.vocab_size, (1, 16))
t0 = time.perf_counter()
logits = max_model(test_ids)
print(f'Forward pass: {(time.perf_counter()-t0)*1000:.0f}ms, shape: {logits.shape}')

# 3 training steps (enough to verify it works at scale)
max_trainer = HybridQuantumTrainer(max_model, learning_rate=1e-4)
ds_max = TextDataset(token_ids, 16)
dl_max = DataLoader(ds_max, 2, shuffle=True)

print('Training 3 steps...')
for i, (x, y) in enumerate(dl_max):
    if i >= 3: break
    t0 = time.perf_counter()
    r = max_trainer.training_step(x, y)
    ms = (time.perf_counter() - t0) * 1000
    print(f'  Step {i+1}: loss={r["loss"]:.4f}, grad_norm={r["grad_norm"]:.4f}, {ms:.0f}ms')

print(f'\nLargest model trainable with quantum circuits on Colab T4.')

In [ ]:
##############################################################################
# STEP 8: FINAL SUMMARY
##############################################################################
print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)
print(f'''
Backend: {BACKEND} ({DIFF_METHOD})

Standard Model (d=32, 1 qhead):
  Loss: {std_data[0]["loss"]:.4f} -> {std_data[-1]["loss"]:.4f} (10 epochs)
  Speed: {std_data[-1]["ms_step"]:.0f} ms/step

Big Model (d=64, 2 layers, 2 qheads, quantum embed):
  Loss: {big_data[0]["loss"]:.4f} -> {big_data[-1]["loss"]:.4f} (10 epochs)
  Speed: {big_data[-1]["ms_step"]:.0f} ms/step

Max Model (d=128, 4 layers): Forward + training verified.

Cross-machine comparison (standard model):
  MacBook i7 (default.qubit):     3534 ms/step
  Linux Desktop (lightning.qubit): 2450 ms/step
  Colab T4:                        {std_data[-1]["ms_step"]:.0f} ms/step

Copyright (c) 2026 Saikrishna Garikipati. All Rights Reserved.
''')